# NECIS Waveform Download Pipeline

In [ ]:
import os, sys, asyncio
from datetime import datetime, date, timedelta
sys.path.insert(0, "/home/msseo/works/Claude")

from necis.config import NECISConfig
from necis.browser import NECISBrowser

cfg = NECISConfig.from_env(headless=True)
print(f"User        : {cfg.username}")
print(f"Download dir: {cfg.download_dir}")


## Step 1+2 — Submit request and fetch (runs automatically)

Submits a download request, then polls every 30 s until the server
prepares the zip (~2–5 min), then downloads it. No manual waiting needed.

Edit `TARGET_DATE`, `STATIONS`, and `COMPONENTS` as needed.

> `STATIONS = None` requests all 404 KS stations (~4 GB/day). Use a short list for testing.

In [ ]:
from pathlib import Path
from necis.continuous import request_day
from necis.fetch_downloads import fetch_ready_downloads
from necis.utils import process_continuous_downloads

TARGET_DATE    = date.today() - timedelta(days=1)  # change as needed
STATIONS       = ["ADOA", "AGSA"]  # None = all 404 stations (~4 GB/day)
COMPONENTS     = ["Z", "N", "E"]             # "E", "N", "Z"
ZIP_DIR        = Path("data/necis/zips")
CONTINUOUS_DIR = Path("/home/msseo/works/Claude/data/necis/continuous")

async def download_one_day():
    async with NECISBrowser(cfg) as nb:
        submitted_after = datetime.now()

        ok = await request_day(nb, day=TARGET_DATE,
                               stations=STATIONS, components=COMPONENTS)
        if not ok:
            print("Request failed — check screenshot in data/necis/")
            return

        print(f"[{TARGET_DATE}] Request submitted. Polling for ready file ...")
        files = await fetch_ready_downloads(
            nb,
            out_dir=ZIP_DIR,
            poll_interval=30,
            max_wait=600,
            submitted_after=submitted_after,
        )
        print(f"Downloaded {len(files)} file(s):")
        for f in files:
            print(f"  {f}  ({f.stat().st_size / 1e6:.1f} MB)")
        return files

files = await download_one_day()

## Step 2 — Extract zips and organize into YYYY/STA/

Destination: `/home/msseo/works/Claude/data/necis/continuous`

In [ ]:
from pathlib import Path
from necis.utils import process_continuous_downloads

ZIP_DIR        = Path("data/necis/zips")
CONTINUOUS_DIR = Path("/home/msseo/works/Claude/data/necis/continuous")

organized = process_continuous_downloads(ZIP_DIR, CONTINUOUS_DIR,
                                         move=True, delete_zip=False)
print(f"Organized {len(organized)} miniSEED file(s) into {CONTINUOUS_DIR}")
for p in organized[:20]:
    print(" ", p)


## Step 3 — Verify miniSEED files with obspy

In [ ]:
import glob
from obspy import read
from pathlib import Path

CONTINUOUS_DIR = Path("/home/msseo/works/Claude/data/necis/continuous")
files = sorted(f for f in glob.glob(str(CONTINUOUS_DIR / "**" / "*"), recursive=True)
               if not __import__('os').path.isdir(f))
print(f"Found {len(files)} file(s) in {CONTINUOUS_DIR}")

if files:
    print("\nReading:", files[0])
    st = read(files[0])
    print(st)
    print("\nFirst trace stats:")
    print(st[0].stats)

## Step 4 — Event page discovery (run once)

Inspect the event waveform form to find real selectors.
After running, update the `ADAPT` constants in [necis/events.py](necis/events.py).

In [ ]:
from necis.browser import NECISBrowser

async def inspect_event_page():
    async with NECISBrowser(cfg) as nb:
        await nb.goto_events()
        await asyncio.sleep(2)
        shot = await nb.screenshot("event_page")
        print("URL:", nb.page.url)
        print("Screenshot:", shot)

        elements = await nb.page.evaluate("""() =>
            ['input','select','button','a'].flatMap(tag =>
                [...document.querySelectorAll(tag)]
                .filter(el => el.offsetParent !== null)
                .map(el => ({
                    tag, type: el.type||'', name: el.name||'', id: el.id||'',
                    text: (el.innerText||'').trim().slice(0,50),
                    onclick: (el.getAttribute('onclick')||'').slice(0,80),
                }))
            )
        """)
        print(f"\n{len(elements)} visible elements:")
        print(f"{'TAG':<8}{'TYPE':<10}{'NAME':<30}{'ID':<25}TEXT/ONCLICK")
        print("-" * 100)
        for el in elements:
            label = el['text'] or el['onclick']
            if label:
                print(f"{el['tag']:<8}{el['type']:<10}{el['name']:<30}{el['id']:<25}{label[:45]}")

await inspect_event_page()
